### 1. Загрузка очищенных данных

Загружаем данные, сохранённые в первом ноутбуке (`data_cleaned_v2.csv`), и проверяем их размер и состав колонок.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка рабочей директории на корень проекта
def setup_project_root():
    current = os.getcwd()
    root = current
    while True:
        if os.path.exists(os.path.join(root, 'data_processed')):
            break
        new_root = os.path.dirname(root)
        if new_root == root:
            root = current
            break
        root = new_root
    os.chdir(root)
    print(f"Корень проекта: {root}")
    return root

PROJECT_ROOT = setup_project_root()

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Загрузка данных
df = pd.read_csv('data_processed/data_cleaned_v2.csv')

print("=== ЗАГРУЗКА ДАННЫХ ===")
print(f"Размер: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"\nКолонки:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

print(f"\nПервые 2 строки:")
print(df.head(2))

Корень проекта: c:\Users\washe\Documents\SF_Training_DS\Диплом
=== ЗАГРУЗКА ДАННЫХ ===
Размер: 356,495 строк, 15 колонок

Колонки:
   1. status
   2. propertyType
   3. street
   4. baths
   5. homeFacts
   6. fireplace
   7. city
   8. schools
   9. sqft
  10. zipcode
  11. beds
  12. state
  13. stories
  14. target
  15. has_fireplace_info

Первые 2 строки:
     status        propertyType             street  baths  \
0    Active  Single Family Home     240 Heather Ln    3.5   
1  for sale  single-family home  12911 E Heroy Ave    2.0   

                                           homeFacts     fireplace  \
0  {'atAGlanceFacts': [{'factValue': '2019', 'fac...      Gas Logs   
1  {'atAGlanceFacts': [{'factValue': '2019', 'fac...  no_fireplace   

             city                                            schools    sqft  \
0  Southern Pines  [{'rating': ['4', '4', '7', 'NR', '4', '7', 'N...  2900.0   
1  Spokane Valley  [{'rating': ['4/10', 'None/10', '4/10'], 'data...  1256.0   

 

### 2. Парсинг homeFacts

**Что такое homeFacts:** Словарь с информацией о строительстве объекта недвижимости.

**Какие признаки извлекаем:**

| Признак | Что означает | Тип |
|---------|-------------|------|
| `year_built` | Год постройки | числовой |
| `price_per_sqft` | Цена за квадратный фут | числовой |
| `heating` | Тип отопления | категориальный |
| `parking` | Тип парковки | категориальный |
| `lot_size` | Размер участка (кв.футы) | числовой |

Эти признаки напрямую влияют на стоимость недвижимости.

**Примечание:** Пропуски в `homeFacts` были заполнены пустым словарем `'{}'` в первом ноутбуке.

In [ ]:
import ast

print("=== ПАРСИНГ HOME FACTS ===\n")

def parse_homefacts(hf_str):
    """Извлекает полезные признаки из homeFacts"""
    if pd.isna(hf_str) or hf_str == '' or hf_str == '{}':
        return {}

    try:
        if isinstance(hf_str, str):
            hf_dict = ast.literal_eval(hf_str)
        else:
            hf_dict = hf_str

        result = {}
        facts = hf_dict.get('atAGlanceFacts', [])

        for fact in facts:
            label = fact.get('factLabel', '')
            value = fact.get('factValue', '')

            if label == 'Year built':
                result['year_built'] = value
            elif label == 'Price/sqft':
                result['price_per_sqft'] = value
            elif label == 'Heating':
                result['heating'] = value
            elif label == 'Parking':
                result['parking'] = value
            elif label == 'lotsize':
                result['lot_size'] = value

        return result
    except:
        return {}

# Применяем парсинг
homefacts_parsed = df['homeFacts'].apply(parse_homefacts)
homefacts_df = pd.json_normalize(homefacts_parsed)
df = pd.concat([df, homefacts_df], axis=1)
df.drop('homeFacts', axis=1, inplace=True)

print(f"✅ Добавлено колонок: {homefacts_df.shape[1]}")
print(f"Новые колонки: {list(homefacts_df.columns)}")

=== ПАРСИНГ HOME FACTS ===

✅ Добавлено колонок: 5
Новые колонки: ['year_built', 'heating', 'parking', 'lot_size', 'price_per_sqft']


In [ ]:
print("=== ЗАПОЛНЕНИЕ ПРОПУСКОВ В ПРИЗНАКАХ ИЗ HOMEFACTS ===\n")

# Сначала преобразуем числовые колонки в правильный тип
homefacts_numeric = ['year_built', 'lot_size']

for col in homefacts_numeric:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        before = df[col].isna().sum()
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"{col}: преобразован в число, заполнено {before} пропусков медианой = {median_val:.1f}")

# price_per_sqft — полностью пустой, удаляем
if 'price_per_sqft' in df.columns:
    df = df.drop('price_per_sqft', axis=1)
    print("price_per_sqft: удалён (все значения NaN)")

# Заполняем категориальные колонки модой
homefacts_categorical = ['heating', 'parking']
for col in homefacts_categorical:
    if col in df.columns:
        before = df[col].isna().sum()
        # Убираем пустые строки из моды
        mode_vals = df[col].mode()
        mode_val = mode_vals[mode_vals != ''].iloc[0] if len(mode_vals[mode_vals != '']) > 0 else 'unknown'
        df[col] = df[col].fillna(mode_val)
        print(f"{col}: заполнено {before} пропусков модой = '{mode_val}'")

=== ЗАПОЛНЕНИЕ ПРОПУСКОВ В ПРИЗНАКАХ ИЗ HOMEFACTS ===

year_built: преобразован в число, заполнено 49740 пропусков медианой = 1985.0
lot_size: преобразован в число, заполнено 324188 пропусков медианой = 9060.0
price_per_sqft: удалён (все значения NaN)
heating: заполнено 3428 пропусков модой = 'unknown'
parking: заполнено 3428 пропусков модой = 'unknown'


### 3. Парсинг schools

**Что такое schools:** Список школ рядом с объектом недвижимости с информацией о рейтинге и расстоянии.

**Какие признаки извлекаем (агрегированные):**

| Признак | Что означает | Тип |
|---------|-------------|------|
| `schools_count` | Количество школ рядом | числовой |
| `avg_school_rating` | Средний рейтинг школ | числовой |
| `nearest_school_dist` | Расстояние до ближайшей школы (мили) | числовой |

Близость к хорошим школам значительно влияет на стоимость недвижимости.

**Примечание:** Пропуски в `schools` были заполнены пустым словарем `'{}'` в первом ноутбуке.

In [ ]:
import re

print("=== ПАРСИНГ SCHOOLS ===\n")

def parse_schools(schools_str):
    """Извлекает агрегированные признаки о школах"""
    if pd.isna(schools_str) or schools_str == '' or schools_str == '{}':
        return {'schools_count': 0, 'avg_school_rating': np.nan, 'nearest_school_dist': np.nan}

    try:
        if isinstance(schools_str, str):
            schools_data = ast.literal_eval(schools_str)
        else:
            schools_data = schools_str

        if not schools_data or len(schools_data) == 0:
            return {'schools_count': 0, 'avg_school_rating': np.nan, 'nearest_school_dist': np.nan}

        ratings = []
        distances = []

        for item in schools_data:
            # Рейтинги
            rating_list = item.get('rating', [])
            for r in rating_list:
                if r and r != 'NR':
                    try:
                        ratings.append(float(r))
                    except:
                        pass

            # Расстояния
            dist_data = item.get('data', {}).get('Distance', [])
            for d in dist_data:
                if d and isinstance(d, str):
                    match = re.search(r'(\d+\.?\d*)', d)
                    if match:
                        distances.append(float(match.group(1)))

        return {
            'schools_count': len(schools_data),
            'avg_school_rating': np.mean(ratings) if ratings else np.nan,
            'nearest_school_dist': min(distances) if distances else np.nan
        }
    except:
        return {'schools_count': 0, 'avg_school_rating': np.nan, 'nearest_school_dist': np.nan}

# Применяем парсинг
schools_parsed = df['schools'].apply(parse_schools)
schools_df = pd.json_normalize(schools_parsed)
df = pd.concat([df, schools_df], axis=1)
df.drop('schools', axis=1, inplace=True)

print(f"✅ Добавлено колонок: {schools_df.shape[1]}")
print(f"Новые колонки: {list(schools_df.columns)}")

=== ПАРСИНГ SCHOOLS ===

✅ Добавлено колонок: 3
Новые колонки: ['schools_count', 'avg_school_rating', 'nearest_school_dist']


In [ ]:
print("=== ЗАПОЛНЕНИЕ ПРОПУСКОВ В ПРИЗНАКАХ ИЗ SCHOOLS ===\n")

schools_cols = ['schools_count', 'avg_school_rating', 'nearest_school_dist']

for col in schools_cols:
    if col in df.columns:
        before = df[col].isna().sum()

        if col == 'schools_count':
            df[col] = df[col].fillna(0)
            print(f"{col}: заполнено {before} пропусков значением 0")
        elif col == 'avg_school_rating':
            # Преобразуем в число
            df[col] = pd.to_numeric(df[col], errors='coerce')
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"{col}: заполнено {before} пропусков медианой = {median_val:.1f}")
        elif col == 'nearest_school_dist':
            # Преобразуем в число
            df[col] = pd.to_numeric(df[col], errors='coerce')
            max_val = df[col].max()
            df[col] = df[col].fillna(max_val)
            print(f"{col}: заполнено {before} пропусков максимальным значением = {max_val:.1f}")

print("\n✅ Пропуски в schools-признаках заполнены")

=== ЗАПОЛНЕНИЕ ПРОПУСКОВ В ПРИЗНАКАХ ИЗ SCHOOLS ===

schools_count: заполнено 0 пропусков значением 0
avg_school_rating: заполнено 252605 пропусков медианой = 5.2
nearest_school_dist: заполнено 3289 пропусков максимальным значением = 1590.4

✅ Пропуски в schools-признаках заполнены


### 4. Нормализация propertyType

**Проблема:** Один и тот же тип недвижимости записан по-разному:
- `mobile`, `mo2le`, `prefab`, `modular` → всё это `manufactured`
- `single family`, `single-family home`, `sfr` → всё это `single-family`
- `townhome`, `town house` → всё это `townhouse`

**Решение:** Приводим все синонимы к единому названию с помощью словаря.

**Словарь синонимов содержит основные типы:**
- `manufactured` (мобильные дома)
- `single-family` (отдельные дома)
- `townhouse` (таунхаусы)
- `condo` (кондоминиумы)
- `apartment` (квартиры)
- `co-op` (кооперативы)
- `cabin` (домики в лесу)
- `ranch`, `craftsman`, `victorian`, `colonial`, `contemporary`, `cottage`, `farmhouse`, `tudor`, `log home` (архитектурные стили)

**Результат:** Количество уникальных значений сократится, что улучшит качество модели.

In [ ]:
print("=== НОРМАЛИЗАЦИЯ PROPERTYTYPE ===\n")

# Словарь синонимов
property_synonyms = {
    # mobile home варианты
    'mobile': 'manufactured',
    'mobile home': 'manufactured',
    'mo2le': 'manufactured',
    'mo2 le': 'manufactured',
    'prefab': 'manufactured',
    'modular': 'manufactured',
    'manufactured': 'manufactured',
    'manufactured home': 'manufactured',
    # cabin варианты
    'cabin': 'cabin',
    'ca2n': 'cabin',
    'ca2 n': 'cabin',
    # midcentury
    'midcentury': 'mid-century',
    'mid century': 'mid-century',
    'mid-century modern': 'mid-century',
    # single family
    'single family': 'single-family',
    'single-family home': 'single-family',
    'single family home': 'single-family',
    'single-family residential': 'single-family',
    'sfr': 'single-family',
    'detached': 'single-family',
    # townhouse
    'townhome': 'townhouse',
    'town house': 'townhouse',
    'townhome/townhouse': 'townhouse',
    # condo
    'condo': 'condo',
    'condominium': 'condo',
    # apartment
    'apartment': 'apartment',
    # co-op
    'co-op': 'co-op',
    'cooperative': 'co-op',
    # архитектурные стили
    'ranch': 'ranch',
    'craftsman': 'craftsman',
    'victorian': 'victorian',
    'colonial': 'colonial',
    'contemporary': 'contemporary',
    'cottage': 'cottage',
    'farmhouse': 'farmhouse',
    'tudor': 'tudor',
    'log home': 'log home'
}

def normalize_property_type(pt):
    """Нормализует тип недвижимости"""
    if pd.isna(pt):
        return np.nan
    pt = str(pt).lower().strip()

    # Прямое попадание
    if pt in property_synonyms:
        return property_synonyms[pt]

    # Поиск по ключевым словам
    for key, value in property_synonyms.items():
        if key in pt:
            return value

    return pt

# Применяем нормализацию
print(f"Уникальных значений ДО: {df['propertyType'].nunique()}")
df['propertyType'] = df['propertyType'].apply(normalize_property_type)
print(f"Уникальных значений ПОСЛЕ: {df['propertyType'].nunique()}")

print("\nРаспределение типов недвижимости:")
print(df['propertyType'].value_counts().head(10))

=== НОРМАЛИЗАЦИЯ PROPERTYTYPE ===

Уникальных значений ДО: 1265
Уникальных значений ПОСЛЕ: 504

Распределение типов недвижимости:
propertyType
single-family    221655
condo             42388
townhouse         26452
lot/land          18512
multi-family       7721
traditional        5910
contemporary       3604
manufactured       3517
coop               3221
multi family       2727
Name: count, dtype: int64


### 5. Нормализация status

**Проблема:** Много статусов продажи с похожим смыслом (например, `for sale`, `Active`, `activated` — всё это активные продажи).

**Решение:** Группируем статусы в несколько категорий:

| Группа | Что означает | Примеры исходных значений |
|--------|-------------|--------------------------|
| `active` | Активно продаётся | 'Active', 'for sale', 'activated' |
| `under_contract` | Под контрактом | 'under contract', 'active under contract' |
| `contingency` | Условная продажа | 'contingency', 'active contingency' |
| `pending` | Ожидает завершения | 'pending', 'pending inspection' |
| `foreclosed` | Банковская недвижимость | 'foreclosed', 'foreclosure' |
| `sold` | Продано | 'sold', 'closed' |
| `other` | Все остальные | — |

**Почему это важно:** Статус продажи может влиять на цену (например, foreclosure часто дешевле).

In [ ]:
print("=== НОРМАЛИЗАЦИЯ STATUS ===\n")

# Группировка статусов
status_groups = {
    'active': [
        'active', 'activated', 'active with contract', 'active with offer',
        'active auction', 'auction active', 'for sale'
    ],
    'under_contract': [
        'under contract', 'under contract showing', 'under contract show',
        'active under contract', 'under contract backups', 'active backup',
        'backup contract', 'pending escape clause', 'pending backup wanted',
        'pending take backups', 'pending continue show'
    ],
    'contingency': [
        'contingency', 'contingency contract', 'active contingency',
        'insp inspection contingency'
    ],
    'pending': [
        'pending', 'pending inspection', 'due diligence period'
    ],
    'foreclosed': [
        'foreclosed', 'foreclosure', 'pre foreclosure', 'pre foreclosure auction'
    ],
    'sold': [
        'sold', 'closed'
    ]
}

def normalize_status(status):
    """Нормализует статус продажи"""
    if pd.isna(status):
        return np.nan
    status_str = str(status).lower().strip()

    for group, keywords in status_groups.items():
        for keyword in keywords:
            if keyword in status_str:
                return group

    return 'other'

# Применяем нормализацию
print(f"Уникальных значений ДО: {df['status'].nunique()}")
df['status'] = df['status'].apply(normalize_status)
print(f"Уникальных значений ПОСЛЕ: {df['status'].nunique()}")

print("\nРаспределение статусов:")
print(df['status'].value_counts())

=== НОРМАЛИЗАЦИЯ STATUS ===

Уникальных значений ДО: 150
Уникальных значений ПОСЛЕ: 7

Распределение статусов:
status
active            329528
other              10816
foreclosed          8615
pending             4761
under_contract      2746
contingency           24
sold                   5
Name: count, dtype: int64


### 6.1 Нормализация street

**Проблема:** Адрес содержит избыточную информацию, создающую уникальность:
- Номер дома (240, 12911, 2005)
- Номер квартиры (#512, Apt 3, Unit 2)
- Более 319,000 уникальных адресов (89.5% от всех записей)

**Решение:** Поэтапная обработка:

1. **Нормализация** — удаляем номер дома и номер квартиры
2. **Группировка редких улиц** — все улицы, встречающиеся менее 50 раз, объединяем в категорию `'other'`
3. **Удаление промежуточных колонок** — оставляем только финальную `street_group`

**Результат:**
- Уникальных значений сокращается с 319,061 до ~192 (191 частая улица + `'other'`)
- Категория `'other'` объединяет ~94% записей с редкими адресами

**Почему это правильно:** Редкие улицы не имеют статистической силы для моделирования, но их объединение сохраняет информацию о том, что адрес не относится к частым.

In [ ]:
import re

print("=== НОРМАЛИЗАЦИЯ STREET (ПОЛНЫЙ ЦИКЛ) ===\n")

def normalize_street(address):
    """Очищает адрес: убирает номер дома и квартиры"""
    if pd.isna(address):
        return np.nan

    address = str(address).lower().strip()

    # Убираем номер квартиры
    address = re.sub(r'#\d+', '', address)
    address = re.sub(r'apt\s+\d+', '', address)
    address = re.sub(r'unit\s+\d+', '', address)
    address = re.sub(r'ste\s+\d+', '', address)

    # Убираем номер дома в начале
    address = re.sub(r'^\d+\s+', '', address)

    # Убираем лишние пробелы
    address = re.sub(r'\s+', ' ', address).strip()

    return address

# 1. Создаём нормализованную колонку
df['street_normalized'] = df['street'].apply(normalize_street)

print(f"Оригинальных улиц: {df['street'].nunique():,}")
print(f"Нормализованных улиц: {df['street_normalized'].nunique():,}")

# 2. Группировка редких улиц
min_occurrences = 50
street_counts = df['street_normalized'].value_counts()
frequent_streets = street_counts[street_counts >= min_occurrences].index.tolist()

df['street_group'] = df['street_normalized'].apply(
    lambda x: x if x in frequent_streets else 'other'
)

print(f"\nПорог: >= {min_occurrences} вхождений")
print(f"Улиц в группе 'other': {(df['street_group'] == 'other').sum():,} ({(df['street_group'] == 'other').sum()/len(df)*100:.2f}%)")
print(f"Уникальных значений в street_group: {df['street_group'].nunique()}")

# 3. Удаляем промежуточные колонки
df = df.drop(columns=['street', 'street_normalized'])

print(f"\n✅ Удалены колонки 'street' и 'street_normalized'")
print(f"✅ Создана колонка 'street_group' с {df['street_group'].nunique()} категориями")
print(f"   Осталось колонок: {df.shape[1]}")

print("\nРаспределение street_group (топ-15):")
for i, (street, count) in enumerate(df['street_group'].value_counts().head(15).items(), 1):
    pct = count / len(df) * 100
    print(f"  {i:2}. {street:35} → {count:5,} ({pct:.2f}%)")

=== НОРМАЛИЗАЦИЯ STREET (ПОЛНЫЙ ЦИКЛ) ===

Оригинальных улиц: 319,061
Нормализованных улиц: 140,593

Порог: >= 50 вхождений
Улиц в группе 'other': 337,545 (94.68%)
Уникальных значений в street_group: 192

✅ Удалены колонки 'street' и 'street_normalized'
✅ Создана колонка 'street_group' с 192 категориями
   Осталось колонок: 20

Распределение street_group (топ-15):
   1. other                               → 337,545 (94.68%)
   2. collins ave                         → 1,076 (0.30%)
   3. brickell ave                        →   682 (0.19%)
   4. address not disclosed               →   670 (0.19%)
   5. biscayne blvd                       →   597 (0.17%)
   6. undisclosed address                 →   512 (0.14%)
   7. s miami ave                         →   404 (0.11%)
   8. (undisclosed address)               →   386 (0.11%)
   9. n bayshore dr                       →   336 (0.09%)
  10. s ocean dr                          →   308 (0.09%)
  11. gulf blvd                           →   224 

### 7. Создание дополнительных признаков

**Какие признаки добавляем и почему:**

| Признак | Формула | Смысл |
|---------|---------|-------|
| `house_age` | 2024 - year_built | Возраст дома. Чем новее дом, тем выше цена |
| `is_land` | 1 если propertyType = 'lot/land' | Флаг земельного участка (у таких объектов нет площади, спален и т.д.) |
| `rooms_per_1000sqft` | (beds + baths) / (sqft / 1000) | Плотность планировки. Эффективность использования площади |

**Почему это важно:** Дополнительные признаки помогают модели улавливать нелинейные зависимости, которые не видны напрямую в исходных данных.

In [ ]:
print("=== СОЗДАНИЕ ДОПОЛНИТЕЛЬНЫХ ПРИЗНАКОВ ===\n")

# 1. Возраст дома (текущий год 2024)
current_year = 2024
if 'year_built' in df.columns:
    # Преобразуем в числовой тип
    df['year_built'] = pd.to_numeric(df['year_built'], errors='coerce')

    df['house_age'] = current_year - df['year_built']
    # Отрицательный возраст (ошибка в данных) заменяем на 0
    df['house_age'] = df['house_age'].clip(lower=0)
    # Заполняем пропуски медианой
    df['house_age'] = df['house_age'].fillna(df['house_age'].median())
    print(f"✅ house_age: создан (диапазон {df['house_age'].min():.0f} ... {df['house_age'].max():.0f} лет)")
else:
    print("⚠️ year_built отсутствует, house_age не создан")

# 2. Флаг земельного участка
if 'propertyType' in df.columns:
    df['is_land'] = df['propertyType'].str.contains('lot|land', case=False, na=False).astype(int)
    land_count = df['is_land'].sum()
    print(f"✅ is_land: создан (земельных участков: {land_count:,} ({land_count/len(df)*100:.2f}%))")
else:
    print("⚠️ propertyType отсутствует, is_land не создан")

# 3. Плотность планировки (комнат на 1000 кв.футов)
if all(col in df.columns for col in ['beds', 'baths', 'sqft']):
    # Преобразуем в числовой тип
    df['beds'] = pd.to_numeric(df['beds'], errors='coerce')
    df['baths'] = pd.to_numeric(df['baths'], errors='coerce')
    df['sqft'] = pd.to_numeric(df['sqft'], errors='coerce')

    # Защита от деления на ноль
    df['rooms_per_1000sqft'] = (df['beds'] + df['baths']) / (df['sqft'] / 1000)
    df['rooms_per_1000sqft'] = df['rooms_per_1000sqft'].replace([np.inf, -np.inf], np.nan).fillna(0)
    print(f"✅ rooms_per_1000sqft: создан (среднее = {df['rooms_per_1000sqft'].mean():.2f})")
else:
    print("⚠️ Не хватает колонок для создания rooms_per_1000sqft")

print(f"\nТекущее количество колонок: {df.shape[1]}")

=== СОЗДАНИЕ ДОПОЛНИТЕЛЬНЫХ ПРИЗНАКОВ ===

✅ house_age: создан (диапазон 0 ... 2023 лет)
✅ is_land: создан (земельных участков: 18,560 (5.21%))
✅ rooms_per_1000sqft: создан (среднее = 4.27)

Текущее количество колонок: 23


### 8. Финальная проверка и сохранение данных

**Проверяем:**
- Отсутствие пропусков
- Типы данных
- Размер датасета

**Сохраняем:** `data_features.csv` для третьего ноутбука (моделирование)

In [ ]:
print("=== ФИНАЛЬНАЯ ПРОВЕРКА И СОХРАНЕНИЕ ===\n")

# Проверка на пропуски
missing = df.isnull().sum().sum()
if missing == 0:
    print("✅ Пропусков в данных нет")
else:
    print(f"⚠️ Осталось пропусков: {missing}")
    print(df.isnull().sum()[df.isnull().sum() > 0])

# Типы данных
print(f"\nТипы данных:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

# Размер
print(f"\nРазмер датасета: {df.shape[0]:,} строк, {df.shape[1]} колонок")

# Сохранение
df.to_csv('data_processed/data_features.csv', index=False)
print(f"\n✅ Сохранено data_processed/data_features.csv")

print("\nСписок всех колонок для моделирования:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

=== ФИНАЛЬНАЯ ПРОВЕРКА И СОХРАНЕНИЕ ===

✅ Пропусков в данных нет

Типы данных:
  status: object
  propertyType: object
  baths: float64
  fireplace: object
  city: object
  sqft: float64
  zipcode: object
  beds: float64
  state: object
  stories: float64
  target: float64
  has_fireplace_info: int64
  year_built: float64
  heating: object
  parking: object
  lot_size: float64
  schools_count: int64
  avg_school_rating: float64
  nearest_school_dist: float64
  street_group: object
  house_age: float64
  is_land: int64
  rooms_per_1000sqft: float64

Размер датасета: 356,495 строк, 23 колонок

✅ Сохранено data_processed/data_features.csv

Список всех колонок для моделирования:
   1. status
   2. propertyType
   3. baths
   4. fireplace
   5. city
   6. sqft
   7. zipcode
   8. beds
   9. state
  10. stories
  11. target
  12. has_fireplace_info
  13. year_built
  14. heating
  15. parking
  16. lot_size
  17. schools_count
  18. avg_school_rating
  19. nearest_school_dist
  20. street_g